# openai_agents smoke test

Smoke test for the OpenAI Agents SDK (PyPI: `openai-agents`, import: `agents`).
No network calls are made — examples cover the local-only API surface (Agent definitions, function tools, handoffs, guardrails, runner config).

In [ ]:
# Example 1: Basic Agent construction.
from agents import Agent

agent = Agent(
    name="Haiku writer",
    instructions="You are a helpful assistant that writes haikus.",
)

print(f"Agent name: {agent.name}")
print(f"Agent instructions: {agent.instructions[:60]}...")
print(f"Tools count: {len(agent.tools)}")
print(f"Handoffs count: {len(agent.handoffs)}")

assert agent.name == "Haiku writer"
assert isinstance(agent.tools, list)
assert isinstance(agent.handoffs, list)

In [ ]:
# Example 2: function_tool decorator — builds a tool schema from a Python function.
from agents import function_tool

@function_tool
def get_weather(city: str, units: str = "celsius") -> str:
    """Get current weather for the given city."""
    return f"The weather in {city} is mild ({units})."

print(f"Tool name: {get_weather.name}")
print(f"Tool description: {get_weather.description}")
print(f"Params JSON schema keys: {sorted(get_weather.params_json_schema.keys())}")

assert get_weather.name == "get_weather"
assert "properties" in get_weather.params_json_schema
assert "city" in get_weather.params_json_schema["properties"]

In [ ]:
# Example 3: Agent with tools + handoffs.
from agents import Agent, function_tool

@function_tool
def lookup_order(order_id: str) -> str:
    """Return order status."""
    return f"Order {order_id}: shipped"

billing_agent = Agent(name="Billing", instructions="Handle billing questions.")
support_agent = Agent(
    name="Support",
    instructions="Answer support questions. Hand off to billing for payment issues.",
    tools=[lookup_order],
    handoffs=[billing_agent],
)

print(f"Support tools: {[t.name for t in support_agent.tools]}")
print(f"Support handoffs: {[h.name if hasattr(h, 'name') else h.agent_name for h in support_agent.handoffs]}")

assert len(support_agent.tools) == 1
assert len(support_agent.handoffs) == 1

In [ ]:
# Example 4: Pydantic-typed structured output via output_type.
from pydantic import BaseModel
from agents import Agent

class TriageResult(BaseModel):
    category: str
    urgency: int
    summary: str

triage_agent = Agent(
    name="Triage",
    instructions="Categorize incoming support tickets.",
    output_type=TriageResult,
)

print(f"Output type: {triage_agent.output_type.__name__}")
print(f"Schema fields: {list(TriageResult.model_fields.keys())}")

assert triage_agent.output_type is TriageResult
assert set(TriageResult.model_fields.keys()) == {"category", "urgency", "summary"}

In [ ]:
# Example 5: RunConfig + ModelSettings construction (no network calls).
from agents import ModelSettings, RunConfig

settings = ModelSettings(temperature=0.2, top_p=0.9, max_tokens=512)
config = RunConfig(model_settings=settings, workflow_name="smoke-test-workflow")

print(f"Model temperature: {config.model_settings.temperature}")
print(f"Model top_p: {config.model_settings.top_p}")
print(f"Workflow name: {config.workflow_name}")

assert config.model_settings.temperature == 0.2
assert config.model_settings.top_p == 0.9
assert config.workflow_name == "smoke-test-workflow"
print("\nopenai_agents smoke test PASSED")